# 循环神经网络
在8.3小节，我们介绍了n元语法模型：$p(x_t|x_{t-1}, \dots, x_{t-n+1})$

但是这样不如使用隐变量模型：
$$
p(x_t|x_{t-1}, \dots, x_{1}) \approx p(x_t|h_{t-1})(t时间步的输出取决于t-1时间步的隐状态)
\\
h_{t} = f(h_{t-1}, x_{t}) （t时间步的隐状态取决于t-1时间步的隐状态和t时间步的输入）
$$
其中，$h_{t-1}$表示的是隐状态，存储了到时间步t-1的所有序列信息，基于当前输入和之前的隐状态计算得出

## 无隐藏状态神经网络

设隐藏层的激活函数是$\phi$,给定一个小批量样本$X \in \mathbb{R}^{(n \times d)}$, 其中批量大小为n输入维度为d，隐藏层的输出$H \in \mathbb{R}^{(n \times h)}$通过如下公式计算：

$$
H = \phi(X \cdot W_{xh} + b_h)
$$

这里相当于置了一个隐藏层的多层感知机

隐藏层权重参数为$W_{xh} \in \mathbb{R}^{(d \times h)}$,偏置参数为$b_h \in \mathbb{R}^{1 \times h}$，隐藏单元的数目为h，求和时可以使用广播机制


隐藏变量H用作输出层的输入，输出层由下式给出：
$$
O = H \cdot W_{hq} + b_q
$$

对于分类问题：可以使用softmax(O)来计算输出类别的概率分布

## 有隐藏状态的循环神经网络

### 无隐藏状态的RNN
相当于MLP，时间步 $t$ 的隐藏层输出 $H$ 仅仅取决于当前的输入 $X$：$$H = \phi(X W_{xh} + b_h)$$

### 有隐藏状态的RNN
在 RNN 中，当前时间步的隐状态 $H_t$ 不仅依赖当前的输入 $X_t$，还依赖前一个时间步传过来的隐状态 $H_{t-1}$。为此，引入了一个新的隐藏层权重 $W_{hh}$
$$H_t = \phi(X_t W_{xh} + H_{t-1} W_{hh} + b_h)$$
需要说明的参数是：h表示隐藏层的数量

随后，根据当前的隐状态计算输出层：$$O_t = H_t W_{hq} + b_q$$

这里我们使用代码说明各个矩阵的维度：

In [1]:
import torch
from d2l import torch as d2l

X, W_xh = torch.normal(0, 1, (3, 1)), torch.normal(0, 1, (1, 4))  
H, W_hh = torch.normal(0, 1, (3, 4)), torch.normal(0, 1, (4, 4))
torch.matmul(X, W_xh) + torch.matmul(H, W_hh)

c:\Users\20249\.conda\envs\test1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([[ 0.3253, -0.1592,  2.9958, -1.0254],
        [-0.7039,  2.4898, -4.7968,  2.6787],
        [-0.1973,  0.9221,  1.4398, -0.1186]])

沿列拼接矩阵X和H，沿行拼接矩阵$W_{xh}, W_{hh}$

In [3]:
print(X,'\n',H,'\n',W_xh,'\n',W_hh)

tensor([[-0.6369],
        [ 1.8963],
        [-0.3763]]) 
 tensor([[-0.5411, -0.3710, -0.9089,  0.4139],
        [ 0.7041,  1.0897, -0.1273, -1.2294],
        [ 0.6310, -0.3138, -0.8336,  1.2025]]) 
 tensor([[ 0.1366, -0.2290, -1.7882,  0.9240]]) 
 tensor([[ 0.0500,  1.5273, -1.1469,  0.8568],
        [-1.6630,  0.6993, -0.1804, -0.1742],
        [-0.1014, -1.2048, -1.0864, -0.1416],
        [-0.6517, -0.7590,  0.4393, -0.4027]])


In [5]:
print(torch.cat((X, H), dim=1))
print(torch.cat((W_xh, W_hh), dim=0))

tensor([[-0.6369, -0.5411, -0.3710, -0.9089,  0.4139],
        [ 1.8963,  0.7041,  1.0897, -0.1273, -1.2294],
        [-0.3763,  0.6310, -0.3138, -0.8336,  1.2025]])
tensor([[ 0.1366, -0.2290, -1.7882,  0.9240],
        [ 0.0500,  1.5273, -1.1469,  0.8568],
        [-1.6630,  0.6993, -0.1804, -0.1742],
        [-0.1014, -1.2048, -1.0864, -0.1416],
        [-0.6517, -0.7590,  0.4393, -0.4027]])


输出拼接之后的矩阵相乘的结果，得到与```torch.matmul(X, W_xh) + torch.matmul(H, W_hh)```结果相同的矩阵

In [6]:
print(torch.matmul(torch.cat((X, H), dim=1), torch.cat((W_xh, W_hh), dim=0)))

tensor([[ 0.3253, -0.1592,  2.9958, -1.0254],
        [-0.7039,  2.4898, -4.7968,  2.6787],
        [-0.1973,  0.9221,  1.4398, -0.1186]])


## 字符级语言模型

给定一个当前字符，希望结合隐状态的信息去预测下一个字符（图片见书p315）

下面给出t=3时间步输出的计算过程

1. input
假设我们要让网络学会写 "machine" 这个词。在时间步 1（$t=1$），输入 $X_1$ 是字符 'm'；在时间步 2（$t=2$），输入 $X_2$ 是字符 'a'；在时间步 3（$t=3$），输入 $X_3$ 是字符 'c'以此类推。

2. hidden layer
 拿时间步 3 举例：当网络看到输入 $X_3$（也就是 'c'）时，它不会孤立地去猜。它取决于当前的输入 $X_3$ 和前一步留下的记忆 $H_2$ ，生成新的记忆 $H_3$。
 $$H_3 = \phi(X_3 W_{xh} + H_2 W_{hh} + b_h)$$

 3. output layer
 对每个时间步的输出层的输出进行softmax操作，然后再用交叉熵损失计算模型输出和标签之间的误差：$$l(\mathbf{y}, \mathbf{\hat{y}}) = - \sum_{j=1}^{q} y_j \log(\hat{y}_j)$$
 
 对于分类问题使用独热编码表示，在计算的时候只要考虑一下正确预测的类别就好了：
 假设字典是 [a, b, c, d]，当前正确的字符是 b，那么真实标签 $\mathbf{y}$ 就是 [0, 1, 0, 0]；即这里只用考虑$\hat{y}_{correct}$

## 困惑度

计算序列的似然概率来衡量模型的质量：
$$
\frac{1}{n} \sum_{t=1}^{n} -logP(x_t|x_{t-1}, \ldots, x_1)
$$

也可以套一个exp函数：1表示模型非常完美（P=1，模型非常确定），无穷大表示最差的效果（P=0，模型非常不确定）

## 梯度裁剪

在训练循环神经网络（RNN）时，使用的是通过时间反向传播，处理长度为T的序列，反向传播的时候会涉及到隐藏状态权重矩阵连续相乘T次

假设我们要更新的整个模型所有参数的梯度拼在一起，形成一个巨大的向量，记作 $\mathbf{g}$。我们人为设定一个阈值（通常是一个超参数，比如 $\theta = 1$ 或 $\theta = 5$）。

$$\mathbf{g} \leftarrow \min\left(1, \frac{\theta}{\Vert{}\mathbf{g}\Vert{}} \right) \mathbf{g}$$

这里只能保证训练的时候不会失败，关于梯度消失问题是无法解决的